In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset, load_dataset_builder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt # for making figures
%matplotlib inline
import tiktoken
import pyarrow as pa
import pyarrow.compute as pc
import os

/home/taihim/projects/YA-GPT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open("../data/pg100.txt", "r") as f:
  shakespeare_text = (f.read())

In [4]:
encoding = tiktoken.get_encoding("cl100k_base")

In [5]:
encoded_text = encoding.encode(shakespeare_text)

In [6]:
n = int(0.9*len(encoded_text))
train_data = torch.tensor(encoded_text[:n])
val_data = torch.tensor(encoded_text[n:])

print(f"Training data shape: {train_data.shape}")
print(f"Val data shape: {val_data.shape}")

Training data shape: torch.Size([1312907])
Val data shape: torch.Size([145879])


In [7]:
context_size = 16
batch_size = 4

In [8]:
torch.manual_seed(1337)
def get_batch(split="train"):
  data = train_data if split=="train" else val_data
  idx = torch.randint(len(data) - context_size, (batch_size,), )
  x = torch.stack([data[i:i+context_size] for i in idx])
  y = torch.stack([data[i+1:i+context_size+1] for i in idx])
  return x, y

xb, yb = get_batch()
print(xb.shape)
print(yb.shape)
print(xb[0])
print(yb[0])

print("-----" * 15)
print("")

for i in range(batch_size):
  for l in range (context_size):
    print(f"When input is {xb[i, :l+1].tolist()}, target is {yb[i, l]}")

torch.Size([4, 16])
torch.Size([4, 16])
tensor([26236, 68603,    26,   369,   433,  5084,   198,  7009, 81413,   311,
          387, 30526,    13, 10699,   374,   420])
tensor([68603,    26,   369,   433,  5084,   198,  7009, 81413,   311,   387,
        30526,    13, 10699,   374,   420,   198])
---------------------------------------------------------------------------

When input is [26236], target is 68603
When input is [26236, 68603], target is 26
When input is [26236, 68603, 26], target is 369
When input is [26236, 68603, 26, 369], target is 433
When input is [26236, 68603, 26, 369, 433], target is 5084
When input is [26236, 68603, 26, 369, 433, 5084], target is 198
When input is [26236, 68603, 26, 369, 433, 5084, 198], target is 7009
When input is [26236, 68603, 26, 369, 433, 5084, 198, 7009], target is 81413
When input is [26236, 68603, 26, 369, 433, 5084, 198, 7009, 81413], target is 311
When input is [26236, 68603, 26, 369, 433, 5084, 198, 7009, 81413, 311], target is 387
Whe

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel,
# treating the heads as another batch dimension


# 64, 256, 512
# 2 head attention -> 64, 256, 256
# after concat -> 64, 256, 512

# how to do this in parallel?
# treat each head as a batch dimension
# if 2 heads
# 64, 2, 256, 256?
# then combine?

# q,k,v wont be small size then i.e. they will be embd_size, embd_size instead of embd_size, embd_size // num_heads
# first we reshape from B, T, C to B, T, num_heads, embd_size // num_heads (i.e. head_size)
# then we transpose num_heads and T dimension to get B, num_heads, T, head_size
# we carry out all operations on this matrix
# at the end we reshape back to B, T, C

import math
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

embd_size = 768
max_len = 256
block_size = 512
num_blocks = 8
dropout = 0.1

class MultiHeadAttention(nn.Module):
  def __init__(self, n_embd, n_heads):
    super().__init__()
    self.n_embd = n_embd
    self.n_heads = n_heads
    self.head_size = n_embd // n_heads

    self.Wk = nn.Linear(n_embd, n_embd)
    self.Wq = nn.Linear(n_embd, n_embd)
    self.Wv = nn.Linear(n_embd, n_embd)

    self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape

    k = self.Wk(x)
    q = self.Wq(x)
    v = self.Wv(x)

    k = k.view(B, T, self.n_heads, self.head_size)
    q = q.view(B, T, self.n_heads, self.head_size)
    v = v.view(B, T, self.n_heads, self.head_size)

    k = k.transpose(2, 1)
    q = q.transpose(2, 1)
    v = v.transpose(2, 1)


    wei = q @ k.transpose(-2, -1) / math.sqrt(self.head_size)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    out = wei @ v

    out = out.transpose(1, 2).reshape(B, T, C)

    out = self.dropout(self.proj(out))

    return out

class Ffwd(nn.Module):
  def __init__(self, embd_size, head_size, n_heads):
    super().__init__()
    # self.ffwd = nn.Sequential(nn.Linear(embd_size, head_size * n_heads), nn.ReLU(), nn.Linear(head_size * n_heads, embd_size), nn.Dropout(dropout))
    # gemini suggested expanding linear layer 512 * 4 = 2048
    # self.ffwd = nn.Sequential(nn.Linear(embd_size, 4 * embd_size), nn.ReLU(), nn.Linear(4 * embd_size, embd_size), nn.Dropout(dropout))
    self.ffwd = nn.Sequential(nn.Linear(embd_size, 4 * embd_size), nn.GELU(), nn.Linear(4 * embd_size, embd_size), nn.Dropout(dropout))

  def forward(self, x):
    return self.ffwd(x)


class Block(nn.Module):
  def __init__(self, embd_size, n_heads):
    super().__init__()
    self.ln1 = nn.LayerNorm(embd_size)
    self.mh_attn = MultiHeadAttention(embd_size, n_heads)
    self.ln2 = nn.LayerNorm(embd_size)
    self.ffwd = Ffwd(embd_size, embd_size // n_heads, n_heads)

  def forward(self, x):
    out = x + self.mh_attn(self.ln1(x))
    out = out + self.ffwd(self.ln2(out))

    return out


class AttentionLanguageModel(nn.Module):
  def __init__(self, vocab_size, n_heads, num_blocks):
    super().__init__()
    self.embedding_table = nn.Embedding(vocab_size, embd_size)
    self.position_embedding_table = nn.Embedding(max_len, embd_size)

    self.blocks = nn.ModuleList([Block(embd_size, n_heads) for _ in range(num_blocks)])

    self.ff = nn.Linear(embd_size, vocab_size)


  def forward(self, idx, target=None):
    _, T = idx.shape
    logits = self.embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

    out = logits
    for block in self.blocks:
      out = block(out)

    out = self.ff(out)

    loss = None
    if not target is None:
      B,T,C = out.shape
      out = out.view(B*T, C)

      targets = target.view(B*T)
      loss = F.cross_entropy(out, targets)

    return out, loss

  def generate(self, ctx, max_tokens):
    for _ in range(max_tokens):
      logits, _ = self(ctx)

      # get only last timestep becuase thats the prediction for whats next
      # we already know the preceding chars

      logits = logits[:, -1, :]

      probs = F.softmax(logits, dim=1)
      generated_token = torch.multinomial(probs, num_samples=1)
      ctx = torch.cat((ctx, generated_token), dim=1)

    return ctx

# m = AttentionLanguageModel(len(pretrain_vocab), 8, num_blocks).to(device)
m = AttentionLanguageModel(encoding.n_vocab, 8, num_blocks).to(device)

xb = xb.to(device)
output, _ = m(xb)
print(output.shape)

torch.Size([4, 16, 100277])


In [11]:
for p in m.parameters():
  print(p.shape)

optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)
eval_iters = 200
def estimate_loss(model):
    device = next(model.parameters()).device
    model.eval()
    losses = {'train': 0.0, 'val': 0.0}
    for split in ['train', 'val']:
        total_loss = 0.0
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            xb = xb.to(device)
            yb = yb.to(device)
            _, loss = model(xb, yb)
            total_loss += loss.item()
        losses[split] = total_loss / eval_iters
    model.train()
    return losses

torch.Size([100277, 768])
torch.Size([256, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([3072, 768])
torch.Size([3072])
torch.Size([768, 3072])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([3072, 768])
torch.Size([3072])
torch.Size([768, 3072])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([3072, 768])
torch.Size([3072])
torch.Size([768, 3072])
torch.

In [12]:
# parallel multi head attn (4 heads, 128 head size, 512 embedding)
# more ram usage but faster training
# 8 epochs in 30mins instead of 7
# 12gb ram instead of 7.5gb
batch_size = 16
epochs = 5000
eval_interval = 500

for i in range(epochs):
  xb, yb = get_batch('train')

  xb = xb.to(device)
  yb = yb.to(device)

  if i % eval_interval == 0:
    losses = estimate_loss(m)
    print(f"Train loss {losses['train']} and eval loss: {losses['val']}")

  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)

  loss.backward()

  optimizer.step()

print(loss)

Train loss 11.911813168525695 and eval loss: 11.921122779846192
Train loss 5.8225343227386475 and eval loss: 6.602016417980194
Train loss 5.405097408294678 and eval loss: 6.387956967353821
Train loss 5.142578122615814 and eval loss: 6.2134927773475646
Train loss 5.050062687397003 and eval loss: 6.106666152477264
Train loss 4.927401478290558 and eval loss: 6.061073794364929
Train loss 4.821320906877518 and eval loss: 6.021613426208496
Train loss 4.766892268657684 and eval loss: 5.975899472236633
Train loss 4.671813426017761 and eval loss: 5.953928241729736
Train loss 4.583029427528381 and eval loss: 5.958554623126983
tensor(4.9775, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
# pretraining gpt generation with tiktoken encodings 1 runs 4.58 train loss 5.95 val loss

# gen_start = torch.zeros((1, 1), dtype=torch.long).to(device)
gen_start = torch.tensor([[198]], dtype=torch.long, device=device)
output = "".join([encoding.decode([idx]) for idx in m.generate(gen_start, max_tokens=256).squeeze().tolist()])
print(output)

In [22]:
import tiktoken

# Replace 'gpt2' with whichever encoding you loaded for your dataset
encoding = tiktoken.get_encoding("cl100k_base") 

# Encode a simple newline character
newline_token = encoding.encode("\n")

print(newline_token) 
# Output: [198]

[198]
